# 02 — Train Language Model

Trains an **LSTM** or **Transformer** language model on grammar strings using next-token prediction.

The trained model and vocabulary are saved as a checkpoint for use in `03_probe.ipynb`.

Key design choices (from Barry's notes):
- CLS and SEP tokens wrap each sequence; PAD tokens are ignored in the loss
- Transformer can be run in **causal** (GPT-style) or **bidirectional** (BERT-style) mode
- LSTM uses `LSTMCell` layer-by-layer so hidden states are accessible for probing

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

from grammar_loader import load_grammar, build_vocab, tokenize

## Configuration

In [ ]:
DATA_FILE       = 'data/anbn_n1-20.txt'   # output of 01_generate_data.ipynb
GRAMMAR_FILE    = 'grammars/anbn.txt'      # grammar file (for vocabulary)
MODEL_TYPE      = 'lstm'                   # 'lstm' or 'transformer'
CAUSAL_MASK     = True                     # Transformer only: True=causal, False=bidirectional

# Hyperparameters
EPOCHS          = 50
BATCH_SIZE      = 32
LR              = 1e-3
EMBED_DIM       = 64
HIDDEN_DIM      = 256
NUM_LAYERS      = 2

CHECKPOINT_DIR  = 'checkpoints'

## Dataset

In [ ]:
class GrammarDataset(Dataset):
    """Loads grammar strings and returns token-ID sequences."""

    def __init__(self, filepath: str, vocab: dict):
        self.vocab = vocab
        self.sequences = []
        with open(filepath) as f:
            for line in f:
                line = line.strip()
                if line:
                    ids = tokenize(line, vocab, add_special=True)
                    self.sequences.append(torch.tensor(ids, dtype=torch.long))

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx]


def collate_fn(batch, pad_id: int):
    """Pad sequences in a batch to the same length."""
    max_len = max(seq.size(0) for seq in batch)
    padded = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(batch):
        padded[i, :seq.size(0)] = seq
    return padded

## Models

In [ ]:
class LSTMLanguageModel(nn.Module):
    """
    LSTM language model built from a stack of LSTMCells.

    Using LSTMCell instead of nn.LSTM gives direct access to each layer's
    hidden state at every timestep, which is required for layer-wise probing.
    """

    def __init__(self, vocab_size: int, embed_dim: int = 64, hidden_dim: int = 256,
                 num_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.cells = nn.ModuleList([
            nn.LSTMCell(embed_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.output_proj = nn.Linear(hidden_dim, vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, x, return_hidden=False):
        """
        x: (batch, seq_len) token IDs
        Returns logits (batch, seq_len, vocab_size).
        If return_hidden=True, also returns a list of (B, T, H) tensors — one per layer.
        """
        B, T = x.shape
        emb = self.embedding(x)                         # (B, T, E)

        h = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        c = [torch.zeros(B, self.hidden_dim, device=x.device) for _ in self.cells]
        layer_outputs = [[] for _ in self.cells]

        for t in range(T):
            inp = emb[:, t, :]                          # (B, E)
            for i, cell in enumerate(self.cells):
                h[i], c[i] = cell(inp, (h[i], c[i]))   # (B, H)
                inp = self.dropout(h[i])
                layer_outputs[i].append(h[i])

        layer_hiddens = [torch.stack(steps, dim=1) for steps in layer_outputs]
        logits = self.output_proj(layer_hiddens[-1])    # (B, T, V)

        if return_hidden:
            return logits, layer_hiddens
        return logits


class TransformerLanguageModel(nn.Module):
    """
    Transformer language model.

    Use causal_mask=True  for autoregressive (GPT-style) training.
    Use causal_mask=False for bidirectional (BERT-style) training.
    Barry's note: test both — bidirectional should handle closing brackets better.
    """

    def __init__(self, vocab_size: int, embed_dim: int = 64, num_heads: int = 4,
                 num_layers: int = 2, ff_dim: int = 256, dropout: float = 0.1,
                 max_seq_len: int = 512, causal_mask: bool = True):
        super().__init__()
        self.causal_mask = causal_mask
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(embed_dim, vocab_size)
        self.num_layers = num_layers

    def forward(self, x, return_hidden=False):
        B, T = x.shape
        positions = torch.arange(T, device=x.device).unsqueeze(0)
        emb = self.embedding(x) + self.pos_embedding(positions)

        pad_mask = (x == 0)
        causal = None
        if self.causal_mask:
            causal = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)

        out = self.transformer(emb, mask=causal, src_key_padding_mask=pad_mask)
        logits = self.output_proj(out)

        if return_hidden:
            return logits, out
        return logits

## Build vocabulary and dataset

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

grammar = load_grammar(GRAMMAR_FILE)
vocab = build_vocab(grammar)
pad_id = vocab['[PAD]']
print(f"Vocabulary ({len(vocab)} tokens): {vocab}")

dataset = GrammarDataset(DATA_FILE, vocab)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, pad_id),
)
print(f"Training on {len(dataset)} sequences")

## Build model

In [ ]:
vocab_size = len(vocab)

if MODEL_TYPE == 'lstm':
    model = LSTMLanguageModel(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=NUM_LAYERS,
    )
else:
    model = TransformerLanguageModel(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        num_heads=4,
        num_layers=NUM_LAYERS,
        ff_dim=HIDDEN_DIM,
        causal_mask=CAUSAL_MASK,
    )

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Model: {MODEL_TYPE}  params: {sum(p.numel() for p in model.parameters()):,}")

## Training loop

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        batch = batch.to(device)
        inputs  = batch[:, :-1]
        targets = batch[:, 1:]

        logits = model(inputs)
        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:>4}/{EPOCHS}  loss: {avg_loss:.4f}")

## Save checkpoint

In [ ]:
Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)

if MODEL_TYPE == 'lstm':
    tag = 'lstm'
else:
    tag = f'transformer_{"causal" if CAUSAL_MASK else "no_causal"}'

checkpoint_path = f"{CHECKPOINT_DIR}/{grammar.name}_{tag}.pt"

torch.save({
    'model_state': model.state_dict(),
    'vocab': vocab,
    'grammar_name': grammar.name,
    'model_type': MODEL_TYPE,
    'causal_mask': CAUSAL_MASK,
    'args': {
        'embed_dim': EMBED_DIM,
        'hidden_dim': HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
    },
}, checkpoint_path)

print(f"Saved: {checkpoint_path}")